# Pipeline GDELT - Traitement des données

Objectif :
- Collecter les données GDELT
- Nettoyer et transformer
- Enrichir les données
- Sauvegarder en format exploitable

In [1]:
#Cellule pour l'importation des bibliothèques
import pandas as pd # Importer pandas pour manipuler les données en tableaux (DataFrames)
import numpy as np # Importer numpy pour les calculs numériques et gestion des valeurs manquantes
from tqdm import tqdm # Importer tqdm pour afficher des barres de progression
import os # Importer os pour interagir avec le système de fichiers (créer dossiers, vérifier tailles)
from pathlib import Path


# Afficher toutes les colonnes sans limites
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100) # Afficher jusqu'à 100 lignes

In [2]:
#Cellule pour le chargement de la dataset brut
# Chemin vers le fichier brut extrait de BigQuery
CHEMIN_RAW = '../data/raw/benin_raw.csv'df_raw = pd.read_csv(CHEMIN_RAW, low_memory=False) # Lire le fichier CSV brut et le stocker dans un DataFrame

print(f" Lignes : {df_raw.shape[0]}") # Afficher le nombre de lignes du DataFrame
print(f" Colonnes : {df_raw.shape[1]}") # Afficher le nombre de colonnes du DataFrame

print(f"\n Colonnes disponibles :")
for i, col in enumerate(df_raw.columns.tolist(), 1):
    print(f"{i:02d}. {col}") # Afficher la liste de toutes les colonnes disponibles avec leurs rangs


 Lignes : 23859
 Colonnes : 31

 Colonnes disponibles :
  01. GLOBALEVENTID
  02. SQLDATE
  03. MONTHYEAR
  04. YEAR
  05. FractionDate
  06. IsRootEvent
  07. ActionGeo_CountryCode
  08. ActionGeo_FullName
  09. ActionGeo_ADM1Code
  10. ActionGeo_Lat
  11. ActionGeo_Long
  12. Actor1Geo_CountryCode
  13. Actor2Geo_CountryCode
  14. Actor1CountryCode
  15. Actor2CountryCode
  16. Actor1Name
  17. Actor2Name
  18. Actor1Type1Code
  19. Actor2Type1Code
  20. Actor1KnownGroupCode
  21. Actor2KnownGroupCode
  22. EventRootCode
  23. EventBaseCode
  24. EventCode
  25. QuadClass
  26. GoldsteinScale
  27. NumMentions
  28. NumSources
  29. NumArticles
  30. AvgTone
  31. SOURCEURL


In [3]:
#Cellule pour le diagnostic de la qualite de la dataset
print("=== VALEURS MANQUANTES ===")

missing = df_raw.isnull().sum() # Compter le nombre de valeurs manquantes (NaN) par colonne

missing_pct = (missing / len(df_raw) * 100).round(2) # Calculer le pourcentage de valeurs manquantes par colonne, arrondi à 2 décimales

rapport_qualite = pd.DataFrame({ 
    'manquants': missing, # Colonne avec le nombre de valeurs manquantes
    'pourcentage': missing_pct # Colonne avec le pourcentage de valeurs manquantes
}).sort_values('pourcentage', ascending=False) # Créer un DataFrame récapitulatif trié du plus au moins de valeurs manquantes

print(rapport_qualite[rapport_qualite['manquants'] > 0]) # Afficher uniquement les colonnes qui ont au moins une valeur manquante

print("\n=== TYPES DE COLONNES ===")
print(df_raw.dtypes) # Afficher le type de données de chaque colonne (int, float, object...)

print("\n=== APERÇU ===")
df_raw.head(3) # Afficher les 3 premières lignes pour vérifier visuellement les données

=== VALEURS MANQUANTES ===
                       manquants  pourcentage
Actor2KnownGroupCode       23603        98.93
Actor1KnownGroupCode       23378        97.98
Actor2Type1Code            16012        67.11
Actor2CountryCode          13814        57.90
Actor1Type1Code            12798        53.64
Actor1CountryCode          11633        48.76
Actor2Geo_CountryCode       7232        30.31
Actor2Name                  7232        30.31
Actor1Geo_CountryCode       2260         9.47
Actor1Name                  2260         9.47

=== TYPES DE COLONNES ===
GLOBALEVENTID              int64
SQLDATE                    int64
MONTHYEAR                  int64
YEAR                       int64
FractionDate             float64
IsRootEvent                int64
ActionGeo_CountryCode        str
ActionGeo_FullName           str
ActionGeo_ADM1Code           str
ActionGeo_Lat            float64
ActionGeo_Long           float64
Actor1Geo_CountryCode        str
Actor2Geo_CountryCode        str
Actor1Count

,GLOBALEVENTID,SQLDATE,MONTHYEAR,YEAR,FractionDate,IsRootEvent,ActionGeo_CountryCode,ActionGeo_FullName,ActionGeo_ADM1Code,ActionGeo_Lat,ActionGeo_Long,Actor1Geo_CountryCode,Actor2Geo_CountryCode,Actor1CountryCode,Actor2CountryCode,Actor1Name,Actor2Name,Actor1Type1Code,Actor2Type1Code,Actor1KnownGroupCode,Actor2KnownGroupCode,EventRootCode,EventBaseCode,EventCode,QuadClass,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,SOURCEURL
0,1218455632,20250101,202501,2025,2025.0027,1,BN,Benin,BN,9.5,2.25,BN,NaN,BEN,NaN,BENIN,NaN,NaN,NaN,NaN,NaN,5,51,51,1,3.4,5,1,5,-7.547170,https://dailypost.ng/2025/01/01/terrorism-alle...
1,1218457982,20250101,202501,2025,2025.0027,0,BN,Benin,BN,9.5,2.25,BN,NaN,BEN,NaN,BENIN,NaN,GOV,NaN,NaN,NaN,1,10,10,1,0.0,10,1,5,-8.482871,https://punchng.com/benin-republic-summons-nig...
2,1218458164,20250101,202501,2025,2025.0027,1,BN,Benin,BN,9.5,2.25,BN,NaN,NaN,NaN,DIPLOMAT,NaN,GOV,NaN,NaN,NaN,4,40,40,1,1.0,3,1,3,-7.843137,https://www.thecable.ng/benin-republic-summons...


In [4]:
#Cellule pour le nettoyage de la dataset
df = df_raw.copy()
# Copier le DataFrame brut pour ne jamais modifier l'original

# ── 1. CORRECTION DES TYPES ───────────────────────────────

df['SQLDATE'] = pd.to_datetime(
    df['SQLDATE'].astype(str),
    format='%Y%m%d',
    errors='coerce'
)
# Convertir SQLDATE en datetime Python
# format='%Y%m%d' : la date est au format YYYYMMDD (ex: 20250427)
# errors='coerce' : les dates invalides deviennent NaT au lieu de planter

df['FractionDate'] = pd.to_numeric(df['FractionDate'], errors='coerce')
# Convertir la date décimale en nombre (ex: 2025.322)

df['GoldsteinScale'] = pd.to_numeric(df['GoldsteinScale'], errors='coerce')
# Convertir le score de stabilité en décimal

df['AvgTone'] = pd.to_numeric(df['AvgTone'], errors='coerce')
# Convertir le ton médiatique en décimal

df['ActionGeo_Lat'] = pd.to_numeric(df['ActionGeo_Lat'], errors='coerce')
# Convertir la latitude en décimal

df['ActionGeo_Long'] = pd.to_numeric(df['ActionGeo_Long'], errors='coerce')
# Convertir la longitude en décimal

df['NumMentions'] = pd.to_numeric(df['NumMentions'], errors='coerce')
# Convertir le nombre de mentions en entier

df['NumSources'] = pd.to_numeric(df['NumSources'], errors='coerce')
# Convertir le nombre de sources en entier

df['NumArticles'] = pd.to_numeric(df['NumArticles'], errors='coerce')
# Convertir le nombre d'articles en entier

df['IsRootEvent'] = pd.to_numeric(df['IsRootEvent'], errors='coerce')
# Convertir IsRootEvent en entier (0 ou 1)

df['QuadClass'] = pd.to_numeric(df['QuadClass'], errors='coerce')
# Convertir QuadClass en entier (1, 2, 3 ou 4)

print(" Types corrigés")

# ── 2. SUPPRESSION DES DOUBLONS ───────────────────────────

nb_avant = len(df)
# Mémoriser le nombre de lignes avant suppression

df = df.drop_duplicates(subset=['GLOBALEVENTID'])
# Supprimer les lignes en double en se basant sur l'identifiant unique

print(f" Doublons supprimés : {nb_avant - len(df):,}")

# ── 3. SUPPRESSION DES DATES INVALIDES ────────────────────

nb_avant = len(df)
df = df.dropna(subset=['SQLDATE'])
# Supprimer les lignes dont la date est invalide ou manquante

print(f" Lignes sans date supprimées : {nb_avant - len(df):,}")

print(f"\n Dataset après nettoyage : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f" Période couverte : {df['SQLDATE'].min().date()} → {df['SQLDATE'].max().date()}")

 Types corrigés
 Doublons supprimés : 0
 Lignes sans date supprimées : 0

 Dataset après nettoyage : 23,859 lignes x 31 colonnes
 Période couverte : 2025-01-01 → 2025-12-31


In [5]:
#Cellule pour l'enrichissement de la dataset
# ── COLONNES TEMPORELLES ──────────────────────────────────

df['mois'] = df['SQLDATE'].dt.month
# Extraire le numéro du mois (1 à 12)

df['trimestre'] = df['SQLDATE'].dt.quarter
# Extraire le trimestre (1 à 4)

df['annee'] = df['SQLDATE'].dt.year
# Extraire l'année

df['mois_annee'] = df['SQLDATE'].dt.to_period('M').astype(str)
# Créer une colonne "YYYY-MM" (ex: "2025-03") pour les graphiques temporels

df['jour_semaine'] = df['SQLDATE'].dt.dayofweek
# Extraire le jour de la semaine (0=Lundi, 6=Dimanche)
# Utile pour détecter des patterns hebdomadaires (Q4)

# ── CATÉGORISATION DU TON MÉDIATIQUE (Q1, Q2, Q3) ────────

def categoriser_ton(tone):
    # Catégoriser le ton médiatique en 4 niveaux
    if pd.isna(tone):
        return 'inconnu'# Valeur manquante
    elif tone > 3:
        return 'tres_positif'# Couverture très favorable
    elif tone > 1:
        return 'positif'# Couverture favorable
    elif tone < -3:
        return 'tres_negatif'# Couverture très défavorable
    elif tone < -1:
        return 'negatif'# Couverture défavorable
    else:
        return 'neutre'# Couverture neutre

df['ton_categorie'] = df['AvgTone'].apply(categoriser_ton)
# Appliquer la fonction à chaque ligne de AvgTone

# ── CATÉGORISATION GOLDSTEIN (Q1, Q3, Q4) ────────────────

def categoriser_goldstein(score):
    # Catégoriser le score de stabilité
    if pd.isna(score):
        return 'inconnu'elif score >= 5:
        return 'tres_cooperatif'# Événement très positif pour la stabilité
    elif score > 0:
        return 'cooperatif'# Événement positif
    elif score == 0:
        return 'neutre'# Événement neutre
    elif score >= -5:
        return 'conflictuel'# Événement négatif
    else:
        return 'tres_conflictuel'# Événement très déstabilisateur

df['goldstein_categorie'] = df['GoldsteinScale'].apply(categoriser_goldstein)
# Appliquer la fonction à chaque ligne de GoldsteinScale

# ── CATÉGORISATION QUADCLASS (Q2, Q5) ─────────────────────

quadclass_labels = {
    1: 'cooperation_verbale', # Déclarations de soutien, promesses
    2: 'cooperation_materielle', # Aide concrète, accords signés
    3: 'conflit_verbal', # Accusations, critiques, menaces
    4: 'conflit_materiel'# Violence, attaques, arrestations
}
df['quadclass_label'] = df['QuadClass'].map(quadclass_labels)
# Remplacer les codes 1/2/3/4 par des labels lisibles

# ── ZONE GÉOGRAPHIQUE NORD/SUD (Q3) ──────────────────────

departements_nord = [# Alibori
    "Banikoara", "Gogounou", "Kandi", "Karimama", "Malanville", "Segbana",

    # Atacora
    "Boukoumbé", "Cobly", "Kérou", "Kouandé", "Matéri", "Natitingou", "Péhunco", "Tanguiéta", "Toucountouna",

    # Donga
    "Bassila", "Copargo", "Djougou", "Ouaké",

    # Borgou
    "Bembèrèkè", "Kalalé", "N'Dali", "Nikki", "Parakou", "Pèrèrè", "Sinendé", "Tchaourou"]
# BN01=Alibori · BN02=Atacora · BN06=Borgou — zone exposée jihadiste

df['zone_benin'] = df['ActionGeo_FullName'].apply(
    lambda x: 'nord' if x in departements_nord else (
        'sud' if pd.notna(x) else 'inconnu')
)
# Classifier chaque événement en zone nord (sécuritaire) ou sud

# ── EXTRACTION DOMAINE SOURCE (Q1) ────────────────────────

def extraire_domaine(url):
    # Extraire le domaine depuis l'URL pour identifier le média
    if pd.isna(url):
        return 'inconnu'try:
        domaine = url.split('/')[2]
        # Prendre la 3ème partie de l'URL (ex: "www.rfi.fr")
        domaine = domaine.replace('www.', '')
        # Supprimer le préfixe "www."return domaine
    except:
        return 'inconnu'df['source_domaine'] = df['SOURCEURL'].apply(extraire_domaine)
# Extraire le domaine de chaque URL source

print(" Colonnes enrichies ajoutées")
print(f" Dataset enrichi : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"\nNouvelles colonnes ajoutées :")
nouvelles = ['mois', 'trimestre', 'annee', 'mois_annee', 'jour_semaine',
             'ton_categorie', 'goldstein_categorie', 'quadclass_label',
             'zone_benin', 'source_domaine']
for col in nouvelles:
    print(f"{col}")

 Colonnes enrichies ajoutées
 Dataset enrichi : 23,859 lignes x 41 colonnes

Nouvelles colonnes ajoutées :
   mois
   trimestre
   annee
   mois_annee
   jour_semaine
   ton_categorie
   goldstein_categorie
   quadclass_label
   zone_benin
   source_domaine


In [6]:
print(" Vérification des assertions qualité...\n")

# Vérifier que les colonnes critiques sont présentes
colonnes_obligatoires = [
    'GLOBALEVENTID', 'SQLDATE', 'AvgTone',
    'GoldsteinScale', 'ActionGeo_CountryCode'
]
for col in colonnes_obligatoires:
    assert col in df.columns, f" Colonne manquante : {col}"# Planter immédiatement si une colonne critique est absente
    print(f"{col}")

# Vérifier le volume
assert len(df) > 100, f" Trop peu d'événements : {len(df)}"
# S'assurer qu'on a suffisamment de données
print(f"Volume suffisant : {len(df):,} événements")

# Vérifier l'absence de doublons
assert df['GLOBALEVENTID'].duplicated().sum() == 0, " Doublons détectés !"
print(f"Aucun doublon")

# Vérifier que toutes les dates sont valides
assert df['SQLDATE'].isna().sum() == 0, " Dates manquantes !"
print(f"Toutes les dates valides")

# Vérifier la période
assert df['SQLDATE'].min().year >= 2025, " Données avant 2025 détectées !"
print(f"Période correcte : à partir de 2025")

# Vérifier les latitudes
lat_valides = df['ActionGeo_Lat'].dropna()
if len(lat_valides) > 0:
    assert lat_valides.between(-90, 90).all(), " Latitudes hors limites !"print(f"Latitudes valides")

print("\n Toutes les assertions passées !")

 Vérification des assertions qualité...

    GLOBALEVENTID
    SQLDATE
    AvgTone
    GoldsteinScale
    ActionGeo_CountryCode
    Volume suffisant : 23,859 événements
    Aucun doublon
    Toutes les dates valides
    Période correcte : à partir de 2025
    Latitudes valides

 Toutes les assertions passées !


In [7]:
# Créer le dossier processed s'il n'existe pas
os.makedirs('../data/processed', exist_ok=True)
# exist_ok=True évite une erreur si le dossier existe déjà

# Sauvegarder les données nettoyées de base
df.to_csv('../data/processed/benin_clean.csv', index=False)
# index=False pour ne pas écrire l'index pandas dans le fichier

# Sauvegarder les données enrichies
df.to_csv('../data/processed/benin_enrichi.csv', index=False)
# Même données mais avec les colonnes calculées

# Sauvegarder en Parquet (plus rapide et plus léger)
df.to_parquet('../data/processed/benin_enrichi.parquet', index=False)
# Format recommandé pour le dashboard Streamlit

print(" benin_clean.csv sauvegardé")
print(" benin_enrichi.csv sauvegardé")
print(" benin_enrichi.parquet sauvegardé")
print(f"\n Taille CSV : {os.path.getsize('../data/processed/benin_enrichi.csv') / 1024:.1f} KB")
print(f" Taille Parquet : {os.path.getsize('../data/processed/benin_enrichi.parquet') / 1024:.1f} KB")

 benin_clean.csv sauvegardé
 benin_enrichi.csv sauvegardé
 benin_enrichi.parquet sauvegardé

 Taille CSV : 7459.6 KB
 Taille Parquet : 1093.5 KB


In [8]:
print("=" * 55)
print(" RAPPORT QUALITE FINAL -- PIPELINE J2")
print("=" * 55)
print(f"Événements traités : {len(df):,}")
print(f"Période couverte : {df['SQLDATE'].min().date()} → {df['SQLDATE'].max().date()}")
print(f"Colonnes totales : {df.shape[1]}")
print(f"dont originales : 31")
print(f"dont enrichies : {df.shape[1] - 31}")

print(f"\n Valeurs manquantes cles :")
cols_a_verifier = ['GoldsteinScale', 'AvgTone', 'ActionGeo_Lat',
                   'Actor1Name', 'ActionGeo_ADM1Code']
for col in cols_a_verifier:
    if col in df.columns:
        n = df[col].isna().sum()
        pct = (n / len(df) * 100).round(1)
        print(f"{col:<25} : {n:,} ({pct}%)")

print(f"\n Ton mediatique :")
print(df['ton_categorie'].value_counts().to_string())

print(f"\n Type d'evenement (Goldstein) :")
print(df['goldstein_categorie'].value_counts().to_string())

print(f"\n QuadClass :")
print(df['quadclass_label'].value_counts().to_string())

print(f"\n Zone geographique :")
print(df['zone_benin'].value_counts().to_string())

print("\n" + "=" * 55)
print(" Pipeline J2 termine avec succes.")
print("=" * 55)

 RAPPORT QUALITE FINAL -- PIPELINE J2
Événements traités     : 23,859
Période couverte       : 2025-01-01 → 2025-12-31
Colonnes totales       : 41
  dont originales      : 31
  dont enrichies       : 10

 Valeurs manquantes cles :
  GoldsteinScale            : 0 (0.0%)
  AvgTone                   : 0 (0.0%)
  ActionGeo_Lat             : 0 (0.0%)
  Actor1Name                : 2,260 (9.5%)
  ActionGeo_ADM1Code        : 0 (0.0%)

 Ton mediatique :
ton_categorie
tres_negatif    8802
tres_positif    4223
neutre          3982
negatif         3817
positif         3035

  Type d'evenement (Goldstein) :
goldstein_categorie
cooperatif          10760
conflictuel          4487
tres_cooperatif      3205
neutre               2846
tres_conflictuel     2561

  QuadClass :
quadclass_label
cooperation_verbale       15199
conflit_materiel           3372
conflit_verbal             2888
cooperation_materielle     2400

 Zone geographique :
zone_benin
sud    23859

 Pipeline J2 termine avec succes.
